 # 配置与导入

In [1]:
import copy
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding
from kernel.palu_attention import apply_rotary_pos_emb
import lm_eval
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import TaskManager
from lm_eval.utils import make_table
# 超参
MODEL_PATH = "Meta-Llama-3-8B-Instruct_ratio-0.7_gs-4-fisher_uniform-whiten"
DATASET_NAME = "wikitext-2-raw-v1"  # wikitext-2-raw-v1
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


 # 评估函数

In [2]:
### PPL 评估
def evaluate_ppl(model, seqlen=2048, device="cuda", nsamples=None, input_ids=None):
    if input_ids is None: raise ValueError("evaluate_ppl now requires pre-tokenized input_ids to be passed in.")
    assert input_ids.dim() == 2, "input_ids must be 2D"

    if isinstance(device, str):
        device = torch.device(device)

    nsamples = input_ids.numel() // seqlen if nsamples is None else nsamples
    model.eval()

    nlls = []
    loss_fct = nn.CrossEntropyLoss()
    with torch.no_grad():
        for i in tqdm(range(nsamples)):
            batch = input_ids[:, (i * seqlen):((i + 1) * seqlen)].to(device)
            outputs = model(batch)
            logits = outputs.logits
            shift_logits = logits[:, :-1, :]
            shift_labels = input_ids[:, (i * seqlen):((i + 1) * seqlen)][:, 1:].to(device)
            loss = loss_fct(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1)
            )
            neg_log_likelihood = loss.float() * seqlen
            nlls.append(neg_log_likelihood)

    ppl = torch.exp(torch.stack(nlls).sum() / (len(nlls) * seqlen)).item()
    return ppl

def example_generation(model, tokenizer, device):
    prompt = "Why research is so hard?"
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=False
        )

    gen_text = tokenizer.decode(gen_ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print("=== Example Prompt ===")
    print(prompt)
    print(gen_text)
    return

### Zero-shot OpenBookQA 准确率评估
def zero_shot_eval(model, tokenizer, tasks, *,
                                      batch_size: int = 8,
                                      max_length: int = 4096,
                                      limit: int | None = None,
                                      return_full: bool = False):
    """
    Same core logic as run_lm_eval.py but uses an already-loaded model/tokenizer.
    - Wraps model/tokenizer with HFLM
    - Runs lm_eval.simple_evaluate on the given tasks
    - Prints the results table and returns results['results'] by default
    - res = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
    """

    # normalize tasks
    task_list = [t.strip() for t in tasks.split(",")] if isinstance(tasks, str) else list(tasks)

    model.seqlen = max_length
    lm_obj = HFLM(pretrained=model, tokenizer=tokenizer, add_bos_token=False, batch_size=batch_size)
    task_manager = TaskManager()

    with torch.no_grad():
        results = lm_eval.simple_evaluate(
            model=lm_obj,
            tasks=task_list,
            task_manager=task_manager,
            log_samples=False,
            limit=limit,
        )

    print(make_table(results))
    return results if return_full else results["results"]


 ## 1) 加载model和dataset

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto", use_cache=False
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("模型已加载。")

# 需要进行 HACK 的层（示例：修改为 [1, 2] 可对第 1、2 层依次处理）
hack_layer_ids = [0]
active_hack_layer_id = hack_layer_ids[0] if len(hack_layer_ids) > 0 else 0

# 存档 original layers
original_layers = {lid: copy.deepcopy(model.model.layers[lid].self_attn) for lid in hack_layer_ids}
print(f"original_layers 已存档: {list(original_layers.keys())}")

# 初始活动层 id（仅用于初始化变量；实际以循环内为准）

# 预缓存测试集以加速 PPL
ds_train = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
ds_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
test_texts = [ex["text"] for ex in ds_test if ex["text"].strip()]
test_text_cat = "\n\n".join(test_texts)
test_tok = tokenizer(test_text_cat, return_tensors="pt")
test_ids_all = test_tok.input_ids
test_ids_all = tokenizer("\n\n".join(ds_test["text"]), return_tensors="pt").input_ids
print("数据集已缓存。")
rotary_full = LlamaRotaryEmbedding(config=model.model.layers[0].self_attn.config).to(device)
# 重置指定层到 original
def reset_model(model, original_layers, layer_id):
    model.model.layers[layer_id].self_attn = copy.deepcopy(original_layers[layer_id])
    layer = model.model.layers[layer_id].self_attn
    print(f"Model reset to original at layer {layer_id}")
    return layer


2025-09-16:17:16:32,314 INFO     [modeling.py:1005] We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

模型已加载。
original_layers 已存档: [0]
数据集已缓存。


 ## 2) 训练辅助函数



In [4]:
def sample_batch(tokenizer, batch_size=8, seq_len=128, device=device):
    texts = []
    while len(texts) < batch_size:
        t = ds_train[np.random.randint(len(ds_train))]["text"].strip()
        if t:
            texts.append(t)
    tok = tokenizer(
        texts, max_length=seq_len, truncation=True, padding="max_length", return_tensors="pt"
    )
    return tok.input_ids.to(device)

@torch.no_grad()
def capture_layer_input_hidden_states(model, input_ids, layer_id, device):
    """Capture the true hidden_states input to the target decoder layer via a forward pre-hook,
    and abort the rest of the forward immediately to save compute.
    """
    captured = {}
    class _StopForward(Exception): pass
    def _pre_hook(module, args):
        captured[layer_id] = args[0].detach()
        raise _StopForward()

    handle = model.model.layers[layer_id].register_forward_pre_hook(_pre_hook)
    assert model.config.use_cache is False, "use_cache must be False"
    model.eval()
    try:
        _ = model(input_ids.to(device))
    except _StopForward:
        pass
    finally:
        handle.remove()

    if layer_id not in captured: raise RuntimeError(f"Failed to capture hidden states at layer {layer_id}")

    return captured[layer_id]  # [B, T, H]

def alignment_loss(model, input_ids, layer_id, original_attn):
    B, T = input_ids.shape
    pos_ids = torch.arange(T, device=device).unsqueeze(0).expand(B, -1)

    # 获取真实输入到该层的 hidden states
    hs = capture_layer_input_hidden_states(model, input_ids, layer_id, device)  # [B, T, H]
    hs = model.model.layers[layer_id].input_layernorm(hs)
    # 计算 cos/sin（按 K 的 head_dim）
    hack_attn = model.model.layers[layer_id].self_attn
    head_dim = hack_attn.head_dim
    num_kv = hack_attn.num_key_value_heads
    dummy = torch.empty(B, num_kv, T, head_dim, device=device, dtype=hs.dtype)
    cos, sin = rotary_full(dummy, pos_ids)  # 形状 [B, T, head_dim]

    # === 目标：PALU 路径（RoPE(x@U@V)) ===
    with torch.no_grad():
        k_lat_palu = original_attn.k_proj.project_to_latent(hs)  # [B, T, total_latent_k]
        k_palu = original_attn.k_proj.reconstruct(k_lat_palu)    # [B, T, num_kv*head_dim]
        k_palu = k_palu.view(B, T, num_kv, head_dim).transpose(1, 2)  # [B, heads, T, head_dim]
        _, k_palu_rope = apply_rotary_pos_emb(None, k_palu, cos, sin)  # 对 key 施加 RoPE

    # === 预测：HACK 路径（RoPE(x@U)@V) ===
    k_lat_hack = hack_attn.k_proj.project_to_latent(hs.float())  # [B, T, total_latent_k]
    latent_dim = k_lat_hack.shape[-1] // num_kv
    k_lat_hack = k_lat_hack.view(B, T, num_kv, latent_dim).transpose(1, 2)  # [B, heads, T, latent_dim]
    # 在 latent 维度上截断 cos/sin 后应用 RoPE
    _, k_lat_hack_rope = apply_rotary_pos_emb(None, k_lat_hack, cos[..., :latent_dim], sin[..., :latent_dim])
    # 重构回 key states
    k_lat_hack_rope = k_lat_hack_rope.transpose(1, 2).reshape(B, T, -1)  # [B, T, total_latent_k]
    k_hack = hack_attn.k_proj.reconstruct(k_lat_hack_rope).view(B, T, num_kv, head_dim).transpose(1, 2)  # [B, heads, T, head_dim]

    # MSE 对齐
    return nn.functional.mse_loss(k_hack, k_palu_rope.float())


 ## 3) 训练循环：优化 U/V（仅 K），并周期性评估整模 PPL

In [5]:
SEQ_LEN = 2048
BATCH_SIZE = 8
NUM_STEPS = 2000
EVAL_EVERY = 200
MAX_TEST_WINDOWS = 10  # 每次快速 PPL 评估用多少个窗口

best_layers = {}

# 逐层进行 HACK/finetune/验证
for active_hack_layer_id in hack_layer_ids:
    print(f"\n===== Processing layer {active_hack_layer_id} =====")

    # 初始 PPL
    base_ppl = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
    print(f"Baseline before finetune (layer={active_hack_layer_id}), PPL: {base_ppl:.4f}")

    # 初始 HACK Attention PPL
    hack_attn = model.model.layers[active_hack_layer_id].self_attn
    original_attn = copy.deepcopy(original_layers[active_hack_layer_id])
    hack_attn.rope_latent = True  # Set HACK Attention RoPE(x@U)@V
    base_ppl_hack = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
    print(f"HACK Attention (layer={active_hack_layer_id}, rope = {hack_attn.rope_latent}) PPL: {base_ppl_hack:.4f}")

    # 训练
    train_params = [hack_attn.k_proj.VT.weight, hack_attn.k_proj.U[0].weight, hack_attn.k_proj.U[1].weight]
    for n, p in hack_attn.named_parameters(): p.requires_grad_(False)
    for p in train_params: p.requires_grad_(True)  # 只训练当前层的 k_proj 中的 U 和 VT
    optimizer = torch.optim.AdamW(train_params, lr=5e-4, weight_decay=1e-6, eps=1e-8)
    print("构建训练目标 train_params")

    loss_hist = []
    ppl_hist = []
    best_ppl = float('inf')
    best_attn = None
    for step in tqdm(range(1, NUM_STEPS + 1), desc=f"Aligning K at layer {active_hack_layer_id} (RoPE latent vs full)"):
        input_ids = sample_batch(tokenizer, BATCH_SIZE, SEQ_LEN, device)
        optimizer.zero_grad(set_to_none=True)
        hack_attn.to(dtype=torch.float32)
        loss = alignment_loss(model, input_ids, active_hack_layer_id, original_attn)
        if torch.isfinite(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(train_params, 0.05)
            optimizer.step()
            loss_hist.append(float(loss.item()))
        hack_attn.to(dtype=torch.float16)
        if step % EVAL_EVERY == 0 or step <= 10:
            ppl =  evaluate_ppl(model, SEQ_LEN, device=device, nsamples=10, input_ids=test_ids_all)
            ppl_hist.append(ppl)
            print(f"Step {step}: align_loss={loss.item():.6e}, evaluate_ppl with 10 samples={ppl:.4f}")
            if ppl < best_ppl:
                best_ppl = ppl
                best_attn = copy.deepcopy(model.model.layers[active_hack_layer_id].self_attn)

    # 结束后做一次完整 PPL（注入最优权重）
    if best_attn is not None:
        model.model.layers[active_hack_layer_id].self_attn = best_attn
    final_ppl = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
    print(f"Final (HACK@layer{active_hack_layer_id}) PPL: {final_ppl:.4f}")
    example_generation(model, tokenizer, device)
    print("\n--------------------------------------------------------------------------------------------------------------------------------\n")

    # 记录最佳层权重
    best_layers[active_hack_layer_id] = copy.deepcopy(model.model.layers[active_hack_layer_id].self_attn)



===== Processing layer 0 =====


100%|██████████████████████████████████████████████████████████████████████████████████| 141/141 [00:27<00:00,  5.05it/s]


Baseline before finetune (layer=0), PPL: 8.7749


100%|██████████████████████████████████████████████████████████████████████████████████| 141/141 [00:27<00:00,  5.09it/s]


HACK Attention (layer=0, rope = True) PPL: 4035.8945
构建训练目标 train_params


...

Step 1800: align_loss=1.032824e-01, evaluate_ppl with 10 samples=10.7615


Aligning K at layer 0 (RoPE latent vs full): 100%|███████████████████████████████████| 2000/2000 [01:10<00:00, 28.55it/s]


Step 2000: align_loss=1.027117e-01, evaluate_ppl with 10 samples=10.7994


100%|██████████████████████████████████████████████████████████████████████████████████| 141/141 [00:28<00:00,  5.01it/s]
/data/fat/hcp/jihao/submodules/transformers/src/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/fat/hcp/jihao/submodules/transformers/src/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Final (HACK@layer0) PPL: 11.1768
=== Example Prompt ===
Why research is so hard?
 (Part 1)
Research can be a daunting task, especially for those who are new to the field. In this series, we'll explore some of the challenges that researchers face and offer some tips on how to overcome them.
In this first installment, we'll look at the sheer volume of information that researchers must sift

--------------------------------------------------------------------------------------------------------------------------------



 ## 评估 PPL & OpenBookQA



In [6]:
###===============ppl评估 & zero-shot OpenBookQA（逐层）===============###
# Original
for layer_id in hack_layer_ids:
    attn_layer = reset_model(model, original_layers, layer_id)
ppl_original = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
res_original = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
print(f"👉🏻👉🏻👉🏻👉🏻Evaluating Original LLAMA: PPL= {ppl_original:.4f}")
example_generation(model, tokenizer, device)
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")

# HACK before finetune
for layer_id in hack_layer_ids:
    attn_layer.rope_latent = True
ppl_hack = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
res_hack = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
print(f"👉🏻👉🏻👉🏻👉🏻Evaluating LLAMA with HACK on layer {hack_layer_ids} before finetune: PPL= {ppl_hack:.4f}")
example_generation(model, tokenizer, device)
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")

# HACK after finetune (inject best)
for layer_id in hack_layer_ids:
    model.model.layers[layer_id].self_attn = best_layers[layer_id]
final_ppl = evaluate_ppl(model, SEQ_LEN, device=device, input_ids=test_ids_all)
res_hack_finetune = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
print(f"👉🏻👉🏻👉🏻👉🏻Evaluating LLAMA with HACK on layer {hack_layer_ids} after finetune: PPL= {final_ppl:.4f}")
example_generation(model, tokenizer, device)
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")


Model reset to original at layer 0


100%|██████████████████████████████████████████████████████████████████████████████████| 141/141 [00:28<00:00,  4.99it/s]
2025-09-16:17:19:55,967 WARNING  [huggingface.py:118] `pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
2025-09-16:17:19:56,012 WARNING  [huggingface.py:433] HF model type is neither marked as CausalLM or Seq2SeqLM.                     This is expected if your model requires `trust_remote_code=True` but may be an error otherwise.
2025-09-16:17:19:56,037 WARNING  [huggingface.py:337] Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
2025-09-16:17:20:00,807 INFO     [evaluator.py:131] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2025-09-16:17:20:10,547 INFO     [task.py:395] Building contexts for openb

|  Tasks   |Version|Filter|n-shot| Metric |Value|   |Stderr|
|----------|------:|------|-----:|--------|----:|---|-----:|
|openbookqa|      1|none  |     0|acc     |0.342|±  |0.0212|
|          |       |none  |     0|acc_norm|0.440|±  |0.0222|

👉🏻👉🏻👉🏻👉🏻Evaluating Original LLAMA: PPL= 8.7749


/data/fat/hcp/jihao/submodules/transformers/src/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/fat/hcp/jihao/submodules/transformers/src/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


=== Example Prompt ===
Why research is so hard?
 (Part 1)
Research can be a daunting task, especially for those who are new to the field. In this series, we'll explore some of the common challenges that researchers face and offer some practical tips for overcoming them.
In this first installment, we'll focus on the importance of clear research questions and the importance

--------------------------------------------------------------------------------------------------------------------------------



...

=== Example Prompt ===
Why research is so hard?
 (Part 1)
Research is hard, and I'm not just talking about the long hours spent in the lab or the frustration of not getting the results you want. I'm talking about the fundamental challenges of doing research, the things that make it difficult to do research in the first place. In this series of posts

--------------------------------------------------------------------------------------------------------------------------------



100%|██████████████████████████████████████████████████████████████████████████████████| 141/141 [00:28<00:00,  5.02it/s]
2025-09-16:17:21:45,650 WARNING  [huggingface.py:118] `pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
2025-09-16:17:21:45,690 WARNING  [huggingface.py:433] HF model type is neither marked as CausalLM or Seq2SeqLM.                     This is expected if your model requires `trust_remote_code=True` but may be an error otherwise.
2025-09-16:17:21:45,718 WARNING  [huggingface.py:337] Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
2025-09-16:17:21:50,573 INFO     [evaluator.py:131] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2025-09-16:17:22:00,669 INFO     [task.py:395] Building contexts for openb

|  Tasks   |Version|Filter|n-shot| Metric |Value|   |Stderr|
|----------|------:|------|-----:|--------|----:|---|-----:|
|openbookqa|      1|none  |     0|acc     |0.306|±  |0.0206|
|          |       |none  |     0|acc_norm|0.396|±  |0.0219|

👉🏻👉🏻👉🏻👉🏻Evaluating LLAMA with HACK on layer [0] after finetune: PPL= 11.1768


/data/fat/hcp/jihao/submodules/transformers/src/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/fat/hcp/jihao/submodules/transformers/src/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


=== Example Prompt ===
Why research is so hard?
 (Part 1)
Research can be a daunting task, especially for those who are new to the field. In this series, we'll explore some of the challenges that researchers face and offer some tips on how to overcome them.
In this first installment, we'll look at the sheer volume of information that researchers must sift

--------------------------------------------------------------------------------------------------------------------------------



 ### Save Best Layer

In [7]:
for layer_id, best_attn in best_layers.items():
    torch.save(best_attn.state_dict(), f"HACKDump/{MODEL_PATH}_layer{layer_id}.pt")